Wildfire Sensor Placement on Neutral Atom QPU

**Objective:** Submit wildfire sensor placement optimization to Pasqal Cloud for execution on real neutral atom hardware (Fresnel QPU) or emulators.

**Cloud Workflow:**
1. Authenticate with Pasqal Cloud SDK
2. Create Pulser sequence from sensor instance
3. Submit batch of jobs (multiple seeds/parameters)
4. Monitor job status and retrieve results
5. Compare cloud results to local simulation

**Available Devices:**
- `FRESNEL` — 100-qubit neutral atom QPU (hardware, requires quota)
- `EMU_FREE` — Free CPU emulator (up to 10 qubits)
- `EMU_TN` — Tensor network emulator (10-100 qubits, paid)
- `EMU_MPS` — Matrix product state emulator (GPU-accelerated, paid)

---

In [1]:
# ── Setup ──────────────────────────────────────────────────────────────────────
import os, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from collections import Counter

# Pasqal Cloud SDK
try:
    from pasqal_cloud import SDK
    from pasqal_cloud.device import EmulatorType, DeviceTypeName
    HAS_CLOUD_SDK = True
    print('✓ Pasqal Cloud SDK available')
except ImportError:
    HAS_CLOUD_SDK = False
    print('⚠ Pasqal Cloud SDK not installed. Install with: pip install pasqal-cloud')

# Pulser for sequence design
try:
    from pulser import Pulse, Sequence, Register
    from pulser.devices import DigitalAnalogDevice
    from pulser.waveforms import RampWaveform, ConstantWaveform
    HAS_PULSER = True
    print('✓ Pulser SDK available')
except ImportError:
    HAS_PULSER = False
    print('⚠ Pulser not installed. Install with: pip install pulser')

# Check for real data
REAL_CSV = "../data/processed/optimized_sensor_network.csv"
HAS_REAL_DATA = os.path.exists(REAL_CSV)

if HAS_REAL_DATA:
    print(f'✓ Real sensor data found: {REAL_CSV}')
else:
    print(f'⚠ Real data not found, will use synthetic')

print('\n── Setup complete ──')

✓ Pasqal Cloud SDK available
✓ Pulser SDK available
✓ Real sensor data found: ../data/processed/optimized_sensor_network.csv

── Setup complete ──


---
## 1. Cloud Authentication

**Setup Instructions:**
1. Sign up at https://portal.pasqal.cloud/
2. Create or join a project
3. Get your project ID from the portal
4. Set environment variables (recommended) or use interactive login

**Security Best Practice:**
```bash
export PASQAL_USERNAME="your_email@example.com"
export PASQAL_PASSWORD="your_password"
export PASQAL_PROJECT_ID="your_project_id"
```

In [2]:
# =========================
# PASQAL CLOUD AUTH (FIXED CHECK)
# =========================
from pasqal_cloud import SDK

USERNAME = "mmei4@hawk.illinoistech.edu"
PROJECT_ID = "235a327e-bc98-46fa-8917-460621780db4"

try:
    sdk = SDK(
        username=USERNAME,
        password=None,   # prompts for password
        project_id=PROJECT_ID
    )

    # Try a simple API call that SHOULD exist
    batches = sdk.get_batches()  # lightweight check

    print("✓ Logged into Pasqal Cloud!")
    # print(f"✓ Found {list(batches)} existing batches")
    batches = list(sdk.get_batches())
    print(len(batches)) #testing simple API call for authetnication, should print a number

    HAS_CLOUD_ACCESS = True

except Exception as e:
    print("✗ Authentication FAILED")
    print("Error:", e)

    HAS_CLOUD_ACCESS = False

Enter your password: ········


✓ Logged into Pasqal Cloud!
3


---
## 2. Instance Generation (Same as Before)

In [3]:
def haversine(lat1, lon1, lat2, lon2):
    """Distance in meters."""
    R = 6371000
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

def generate_instance(N, K, output_path=None, use_real_data=True):
    """Generate instance from real CSV or synthetic."""
    
    if use_real_data and HAS_REAL_DATA:
        df = pd.read_csv(REAL_CSV)
        if len(df) < N:
            print(f'    CSV has only {len(df)} rows, using synthetic')
            use_real_data = False
        else:
            df = df.head(N).copy()
            lats = df['latitude'].values
            lons = df['longitude'].values
            
            if 'risk_score_norm' in df.columns:
                risk_norm = df['risk_score_norm'].values
            else:
                base = np.random.uniform(0.3, 0.7, N)
                risk_norm = base / base.max()
            
            sensor_ids = df['sensor_id'].values if 'sensor_id' in df.columns else [f'SENSOR_{i:02d}' for i in range(N)]
            layers = df['layer'].values
            risk_source = "REAL - from CSV"
    
    if not use_real_data or not HAS_REAL_DATA:
        np.random.seed(42)
        lats = np.random.uniform(19.03, 19.18, N)
        lons = np.random.uniform(-104.35, -104.15, N)
        base = np.random.uniform(0.2, 0.7, N)
        risk_norm = base / base.max()
        sensor_ids = [f'SENSOR_{i:02d}' for i in range(N)]
        layers = ['Asset_Defense'] * (N//2) + ['Wildland_Perimeter'] * (N - N//2)
        risk_source = "SYNTHETIC"
    
    # Distance matrix
    dist_matrix = np.zeros((N, N))
    for i in range(N):
        for j in range(i+1, N):
            d = haversine(lats[i], lons[i], lats[j], lons[j])
            dist_matrix[i, j] = d
            dist_matrix[j, i] = d
    
    instance = {
        "metadata": {
            "instance_name": f"pasqal_cloud_N{N}_K{K}",
            "created_date": datetime.now().strftime("%Y-%m-%d"),
            "platform": "Pasqal Cloud",
            "risk_source": risk_source
        },
        "problem": {
            "N": N,
            "K_budget": K,
            "min_separation_m": 2000,
            "lambda_budget": 8.0,
            "lambda_spatial": 12.0
        },
        "candidates": [],
        "distance_matrix": dist_matrix.tolist()
    }
    
    for i in range(N):
        instance['candidates'].append({
            "id": i,
            "sensor_id": str(sensor_ids[i]),
            "latitude": float(lats[i]),
            "longitude": float(lons[i]),
            "layer": str(layers[i]),
            "risk_score_norm": float(risk_norm[i])
        })
    
    if output_path:
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        with open(output_path, 'w') as f:
            json.dump(instance, f, indent=2)
        print(f'  Saved: {output_path}')
    
    print(f'  N={N}, K={K}, Source={risk_source}')
    return instance

print('✓ Instance generator defined')

✓ Instance generator defined


---
## 3. Coordinate Transformation & Register Creation

In [4]:
def latlon_to_cartesian(lats, lons):
    """Convert lat/lon to local Cartesian (meters)."""
    lat0, lon0 = np.mean(lats), np.mean(lons)
    m_per_deg_lat = 111320
    m_per_deg_lon = 111320 * np.cos(np.radians(lat0))
    x = (lons - lon0) * m_per_deg_lon
    y = (lats - lat0) * m_per_deg_lat
    return x, y

def scale_to_pulser_register(x_m, y_m, target_span_um=50.0):
    """Scale Cartesian (m) to Pulser register (μm)."""
    x_m = x_m - np.mean(x_m)
    y_m = y_m - np.mean(y_m)
    span_m = max(np.ptp(x_m), np.ptp(y_m))
    scale = target_span_um / (span_m * 1e6)
    x_um = x_m * 1e6 * scale
    y_um = y_m * 1e6 * scale
    m_per_um = span_m / target_span_um
    return x_um, y_um, m_per_um

def instance_to_register(instance, target_span_um=50.0):
    """Convert instance to Pulser Register."""
    lats = np.array([c['latitude'] for c in instance['candidates']])
    lons = np.array([c['longitude'] for c in instance['candidates']])
    x_m, y_m = latlon_to_cartesian(lats, lons)
    x_um, y_um, m_per_um = scale_to_pulser_register(x_m, y_m, target_span_um)
    
    coords = {f"q{i}": (x_um[i], y_um[i]) for i in range(len(lats))}
    register = Register(coords)
    
    min_sep = instance['problem']['min_separation_m']
    blockade_radius_um = min_sep / m_per_um
    
    print(f'  Register: {len(coords)} atoms')
    print(f'  Span: {np.ptp(x_um):.1f} × {np.ptp(y_um):.1f} μm')
    print(f'  Blockade radius ({min_sep}m): {blockade_radius_um:.2f} μm')
    
    return register, blockade_radius_um, m_per_um

print('✓ Coordinate transformation defined')

✓ Coordinate transformation defined


---
## 4. Cloud-Ready Pulse Sequence Builder

In [5]:
def build_cloud_sequence(instance, target_span_um=50.0, duration_us=3.0, max_rabi_MHz=2.0):
    """
    Build sequence with safe parameters that definitely work with DigitalAnalogDevice.
    Uses hardcoded safe values based on DigitalAnalogDevice specifications.
    """
    if not HAS_PULSER:
        raise ImportError('Pulser not installed')
    
    # Build register
    register, blockade_radius_um, m_per_um = instance_to_register(instance, target_span_um)
    
    # Device
    device = DigitalAnalogDevice
    
    # Create sequence
    seq = Sequence(register, device)
    seq.declare_channel('global', 'rydberg_global')
    
    # FIXED: Hardcoded safe values from DigitalAnalogDevice specs
    # DigitalAnalogDevice specs:
    #   max_amp = 62.83 rad/μs (~10 MHz)
    #   max_abs_detuning = 125.66 rad/μs (~20 MHz)
    max_rabi_rad_us = min(max_rabi_MHz * 2 * np.pi, 50.0)  # Cap at ~8 MHz
    max_detuning_rad_us = 100.0  # ~16 MHz, safely within 125.66 limit
    
    # Phase 1: Ramp up Rabi (20% of duration)
    ramp_duration = int(duration_us * 1000 * 0.2)
    omega_ramp = RampWaveform(ramp_duration, 0, max_rabi_rad_us)
    delta_initial = -max_detuning_rad_us
    delta_ramp = ConstantWaveform(ramp_duration, delta_initial)
    seq.add(Pulse(omega_ramp, delta_ramp, 0), 'global')
    
    # Phase 2: Main sweep - from negative to positive detuning (60% of duration)
    sweep_duration = int(duration_us * 1000 * 0.6)
    omega_const = ConstantWaveform(sweep_duration, max_rabi_rad_us)
    delta_final = max_detuning_rad_us * 0.3
    delta_sweep = RampWaveform(sweep_duration, delta_initial, delta_final)
    seq.add(Pulse(omega_const, delta_sweep, 0), 'global')
    
    # Phase 3: Ramp down Rabi (20% of duration)
    rampdown_duration = int(duration_us * 1000 * 0.2)
    omega_down = RampWaveform(rampdown_duration, max_rabi_rad_us, 0)
    delta_down = ConstantWaveform(rampdown_duration, delta_final)
    seq.add(Pulse(omega_down, delta_down, 0), 'global')
    
    print(f'  Sequence: {duration_us:.2f} μs, Ω_max={max_rabi_rad_us/(2*np.pi):.2f} MHz')
    print(f'  Detuning sweep: {delta_initial/(2*np.pi):.2f} → {delta_final/(2*np.pi):.2f} MHz')
    print(f'  Register: {len(register.qubits)} atoms')
    
    metadata = {
        'duration_us': duration_us,
        'max_rabi_MHz': max_rabi_MHz,
        'n_atoms': len(register.qubits)
    }
    
    return seq, metadata

---
## 5. Cloud Job Submission

**Job Parameters:**
- `runs`: Number of measurement shots (1000-10000)
- `device_type`: Target device (FRESNEL, EMU_FREE, EMU_TN, EMU_MPS)
- `tags`: Custom tags for organization
- `wait`: Whether to block until completion

In [6]:
def submit_to_cloud(instance, device_type='EMU_FREE', runs=1000, wait=True, tags=None):
    """Submit job to Pasqal Cloud with safe sequence parameters."""
    if not HAS_CLOUD_ACCESS:
        raise RuntimeError('Cloud access not configured. Check authentication.')
    
    print(f'\n═══ Submitting to Pasqal Cloud ═══')
    print(f'Device: {device_type}')
    print(f'Runs: {runs}')
    print(f'Instance: N={instance["problem"]["N"]}, K={instance["problem"]["K_budget"]}')
    
    # FIXED: Use safe sequence builder with appropriate parameters
    # Adjust duration based on atom count to avoid simulation timeouts
    n_atoms = instance["problem"]["N"]
    if n_atoms <= 10:
        duration_us = 3.0
    elif n_atoms <= 15:
        duration_us = 2.5
    else:
        duration_us = 2.0
    
    max_rabi_MHz = 2.0
    
    seq, metadata = build_cloud_sequence(instance, duration_us=duration_us, max_rabi_MHz=max_rabi_MHz)
    
    # Serialize for cloud
    serialized_seq = seq.to_abstract_repr()
    
    # FIXED: Proper device type mapping
    device_map = {
        'FRESNEL': DeviceTypeName.FRESNEL,
        'EMU_FREE': EmulatorType.EMU_FREE,
        'EMU_TN': EmulatorType.EMU_TN,
        'EMU_MPS': EmulatorType.EMU_MPS,
    }
    device_enum = device_map.get(device_type)
    
    if device_enum is None:
        raise ValueError(f'Unknown device type: {device_type}')
    
    # Create job
    job = {"runs": runs}
    
    if tags is None:
        tags = ["wildfire_sensors", f"N{instance['problem']['N']}", f"K{instance['problem']['K_budget']}"]
    
    try:
        print('\nSubmitting batch...')
        batch = sdk.create_batch(
            serialized_seq,
            [job],
            device_type=device_enum,
            tags=tags,
            wait=wait
        )
        
        print(f'✓ Batch ID: {batch.id}')
        print(f'  Status: {batch.status}')
        
        if wait:
            print('\n  Waiting for results...')
            # Force refresh to get updated status
            batch.refresh()
            print(f'  Final status: {batch.status}')
            
            for job_obj in batch.ordered_jobs:
                print(f'\n  Job {job_obj.id}:')
                print(f'    Status: {job_obj.status}')
                if hasattr(job_obj, 'result') and job_obj.result:
                    print(f'    Result available: {len(job_obj.result)} measurements')
        else:
            print('\n  Job submitted (not waiting for completion)')
            print(f'  Check status: batch.get_status() or batch.wait()')
        
        return batch
    
    except Exception as e:
        print(f'\n✗ Submission failed: {e}')
        raise

---
## 6. Result Processing

In [7]:
def process_cloud_results(batch, instance):
    """
    Extract solution from Pasqal Cloud results.
    
    Returns:
        result dict with coverage, violations, selected sensors
    """
    N = instance['problem']['N']
    K = instance['problem']['K_budget']
    risk = np.array([c['risk_score_norm'] for c in instance['candidates']])
    dist = np.array(instance['distance_matrix'])
    min_sep = instance['problem']['min_separation_m']
    
    # Get results from first job
    job = batch.ordered_jobs[0]
    
    if not job.result:
        return {'error': 'No result available', 'status': job.status}
    
    # Parse measurements
    # Pasqal returns dict: {bitstring: count}
    counts = Counter()
    for measurement in job.result:
        # measurement format depends on device
        # Typically: list of 0/1 or bitstring
        if isinstance(measurement, (list, tuple)):
            bitstring = ''.join(str(int(b)) for b in measurement)
        else:
            bitstring = str(measurement)
        counts[bitstring] += 1
    
    # Find best solution
    best_bitstring = None
    best_score = -np.inf
    
    for bitstring, count in counts.most_common(50):
        if len(bitstring) != N:
            continue
        
        x = np.array([int(b) for b in bitstring], dtype=int)
        coverage = float(np.sum(risk * x))
        
        # Penalties
        budget_penalty = abs(x.sum() - K) * 0.1
        
        selected = np.where(x == 1)[0]
        viol = 0
        for i in selected:
            for j in selected:
                if i < j and 0 < dist[i, j] < min_sep:
                    viol += 1
        
        score = coverage - budget_penalty - viol * 0.05
        
        if score > best_score:
            best_score = score
            best_bitstring = bitstring
    
    if best_bitstring is None:
        return {'error': 'No valid solution found'}
    
    # Extract final solution
    x_best = np.array([int(b) for b in best_bitstring], dtype=int)
    selected = np.where(x_best == 1)[0]
    coverage = float(np.sum(risk[selected]))
    
    viol = 0
    for i in selected:
        for j in selected:
            if i < j and 0 < dist[i, j] < min_sep:
                viol += 1
    
    return {
        'coverage': coverage,
        'violations': int(viol),
        'n_selected': int(x_best.sum()),
        'selected_indices': selected.tolist(),
        'bitstring': best_bitstring,
        'total_counts': sum(counts.values()),
        'unique_bitstrings': len(counts),
        'top_5_counts': dict(counts.most_common(5))
    }

print('✓ Result processing defined')

✓ Result processing defined


---
## 7. Example: Submit N=10 Instance to Cloud

In [8]:
# Generate instance
os.makedirs('cloud_results/instances', exist_ok=True)
instance_path = 'cloud_results/instances/pasqal_cloud_N10_K4.json'

print('\n═══ Generating Instance ═══')
instance = generate_instance(N=12, K=4, output_path=instance_path, use_real_data=True)

# Submit to cloud (if authenticated)
if HAS_CLOUD_ACCESS:
    try:
        batch = submit_to_cloud(
            instance,
            device_type='EMU_FREE',  # Free emulator (up to 12 qubits)
            runs=1000,
            wait=True,
            tags=['test_run', 'N10_K4']
        )
        
        # Process results
        print('\n═══ Processing Results ═══')
        result = process_cloud_results(batch, instance)
        
        if 'error' not in result:
            print(f"\n✓ Solution found:")
            print(f"  Coverage: {result['coverage']:.4f}")
            print(f"  Violations: {result['violations']}")
            print(f"  Selected: {result['n_selected']}/{instance['problem']['K_budget']} sensors")
            print(f"  Bitstring: {result['bitstring']}")
            print(f"  Total measurements: {result['total_counts']}")
            print(f"  Unique bitstrings: {result['unique_bitstrings']}")
        else:
            print(f"\n✗ Error: {result['error']}")
        
        # Save results
        os.makedirs('cloud_results/results', exist_ok=True)
        result_path = f'cloud_results/results/N10_K4_batch_{batch.id}.json'
        with open(result_path, 'w') as f:
            json.dump({
                'batch_id': batch.id,
                'status': batch.status,
                'instance': instance['metadata'],
                'result': result
            }, f, indent=2)
        print(f"\n✓ Results saved: {result_path}")
        
    except Exception as e:
        print(f"\n✗ Cloud execution failed: {e}")
        import traceback
        traceback.print_exc()
else:
    print('\n⚠ Skipping cloud submission (not authenticated)')
    print('  Set credentials to run on Pasqal Cloud')


═══ Generating Instance ═══
  Saved: cloud_results/instances/pasqal_cloud_N10_K4.json
  N=12, K=4, Source=REAL - from CSV

═══ Submitting to Pasqal Cloud ═══
Device: EMU_FREE
Runs: 1000
Instance: N=12, K=4
  Register: 12 atoms
  Span: 29.6 × 50.0 μm
  Blockade radius (2000m): 10.49 μm
  Sequence: 2.50 μs, Ω_max=2.00 MHz
  Detuning sweep: -15.92 → 4.77 MHz
  Register: 12 atoms

Submitting batch...
✓ Batch ID: 6f3ec5e2-f8b7-45a5-8c83-9ea72d7584a8
  Status: DONE

  Waiting for results...
  Final status: DONE

  Job 4cedfd58-9e00-4d27-bf09-f7a1ce9c0cd0:
    Status: DONE
    Result available: 57 measurements

═══ Processing Results ═══

✓ Solution found:
  Coverage: 6.9999
  Violations: 5
  Selected: 9/4 sensors
  Bitstring: 101100111111
  Total measurements: 57
  Unique bitstrings: 57

✓ Results saved: cloud_results/results/N10_K4_batch_6f3ec5e2-f8b7-45a5-8c83-9ea72d7584a8.json


---
## 8. Advanced: Submit to Fresnel QPU (Hardware)

**Requirements:**
- Approved quota for Fresnel access
- N ≤ 100 (current Fresnel limit)
- May incur billing charges

**Uncomment and run after quota approval:**

In [9]:
# # Generate larger instance for hardware
# instance_hardware = generate_instance(N=20, K=8, output_path='cloud_results/instances/fresnel_N20_K8.json')

# # Submit to Fresnel QPU
# if HAS_CLOUD_ACCESS:
#     batch_hardware = submit_to_cloud(
#         instance_hardware,
#         device_type='FRESNEL',  # Real quantum hardware!
#         runs=5000,  # More shots for hardware noise
#         wait=True,
#       
#     )
#     
#     result_hardware = process_cloud_results(batch_hardware, instance_hardware)
#     print(f"\nFresnel QPU Result:")
#     print(f"  Coverage: {result_hardware['coverage']:.4f}")
#     print(f"  Violations: {result_hardware['violations']}")

print('\n⚠ Fresnel submission commented out (requires quota approval)')
print('  Uncomment code above after getting hardware access')


⚠ Fresnel submission commented out (requires quota approval)
  Uncomment code above after getting hardware access


---
## 10. Key Takeaways

### What You Just Did
 Authenticated with Pasqal Cloud  
 Created atom register from real sensor coordinates  
 Built adiabatic optimization pulse sequence  
 Submitted job to cloud (emulator or QPU)  
 Retrieved and processed quantum measurement results  

### Device Comparison
| Device | Qubits | Speed | Cost | Use Case |
|--------|--------|-------|------|----------|
| **EMU_FREE** | ≤10 | Slow | Free | Testing, small demos |
| **EMU_TN** | 10-100 | Medium | Paid | Development, N=20-50 |
| **EMU_MPS** | 10-100 | Fast (GPU) | Paid | Production simulations |
| **FRESNEL** | ≤100 | Fast | Paid | Real quantum advantage |

### Pasqal Cloud Resources
- Portal: https://portal.pasqal.cloud/
- Docs: https://docs.pasqal.com/cloud/
- Pricing: https://portal.pasqal.cloud/offers
- Support: https://portal.pasqal.cloud/contact

---